PCE Modelleme Calışmaları özeti

Bu notebook, bu klasorde yaptigimiz tum modelleme adimlarini, metrikleri ve karsilastirmalari tek yerde toplar.


1. Kurulum ve Dosyalar

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid', context='notebook')
BASE = Path('.')
DATA = BASE / 'data'

files = sorted([p.name for p in DATA.glob('*')])
print('Toplam dosya:', len(files))
for f in files:
    print('-', f)



2. Baslangic Veri Kontrolu

Hazir veri seti: `spacelis_model_hazir_veri.csv`

In [ ]:
df = pd.read_csv(DATA / 'spacelis_model_hazir_veri.csv')
print('Shape:', df.shape)
print('Target var mi?:', 'JV_default_PCE' in df.columns)
print('Duplicate satir:', int(df.duplicated().sum()))
print('Tumden NaN sutun sayisi:', int((df.isna().all()).sum()))

target = df['JV_default_PCE']
print('
Target ozet:')
print(target.describe())


In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['JV_default_PCE'], bins=60, kde=True)
plt.title('JV_default_PCE Dagilimi')
plt.xlabel('PCE')
plt.ylabel('Frekans')
plt.tight_layout()
plt.show()


3. Leakage Etkisi (Neden %99 Yaniltici Olabiliyor?)

In [ ]:
leakage_notes = pd.DataFrame([
    {'Senaryo': 'Leakage benzeri sutunlar dahil', 'R2': 0.9898},
    {'Senaryo': 'Leakage sutunlari cikartildi', 'R2': 0.6228},
])
leakage_notes


In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(data=leakage_notes, x='Senaryo', y='R2', palette='Set2')
plt.ylim(0,1.05)
plt.title('Leakage Etkisi: R2 Kiyas')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


4. Ana Pipeline Sonucu (`train_pce_pipeline.py`)

In [ ]:
report_path = DATA / 'pce_model_report.txt'
if report_path.exists():
    print(report_path.read_text(encoding='utf-8'))
else:
    print('Rapor bulunamadi:', report_path)


In [ ]:
fi_path = DATA / 'pce_feature_importance_top30.csv'
if fi_path.exists():
    fi = pd.read_csv(fi_path)
    display(fi.head(10))
    plt.figure(figsize=(9,5))
    sns.barplot(data=fi.head(10), x='importance', y='feature', palette='viridis')
    plt.title('Top 10 Feature Importance (RandomForest)')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance dosyasi yok:', fi_path)


5. Model Benchmark Sonuclari (RF, ExtraTrees, XGBoost, CatBoost, ... )

In [ ]:
bench_path = DATA / 'pce_model_benchmark_results.csv'
if bench_path.exists():
    bench = pd.read_csv(bench_path)
    display(bench)
else:
    bench = None
    print('Benchmark dosyasi bulunamadi:', bench_path)


In [ ]:
if bench is not None:
    plt.figure(figsize=(8,4))
    tmp = bench.sort_values('cv_r2_mean', ascending=False)
    sns.barplot(data=tmp, x='model', y='cv_r2_mean', palette='Blues_d')
    plt.title('Modellerin CV R2 Kiyaslamasi')
    plt.ylabel('CV R2 (mean)')
    plt.xlabel('Model')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,4))
    tmp2 = bench.sort_values('holdout_r2', ascending=False)
    sns.barplot(data=tmp2, x='model', y='holdout_r2', palette='Greens_d')
    plt.title('Modellerin Holdout R2 Kiyaslamasi')
    plt.ylabel('Holdout R2')
    plt.xlabel('Model')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


6. Tuning Sonuclari (`tune_pce_models_fast.py`)

In [ ]:
tuned_path = DATA / 'tuned_pce_results.csv'
if tuned_path.exists():
    tuned = pd.read_csv(tuned_path)
    display(tuned)
else:
    tuned = None
    print('Tuned sonuc dosyasi bulunamadi:', tuned_path)


In [ ]:
if tuned is not None:
    melted = tuned.melt(id_vars=['model'], value_vars=['cv_r2_best','holdout_r2'], var_name='metric', value_name='value')
    plt.figure(figsize=(7,4))
    sns.barplot(data=melted, x='model', y='value', hue='metric', palette='Set1')
    plt.title('Tuned Modeller: CV vs Holdout R2')
    plt.ylabel('R2')
    plt.xlabel('Model')
    plt.tight_layout()
    plt.show()


## 7. V2 Group CatBoost Sonuclari (Ham Veri Tabanli)

In [ ]:
v2_path = DATA / 'v2_group_catboost_cv_results.csv'
if v2_path.exists():
    v2 = pd.read_csv(v2_path)
    display(v2)
else:
    v2 = None
    print('V2 CV dosyasi bulunamadi:', v2_path)

v2_report = DATA / 'v2_group_catboost_report.txt'
if v2_report.exists():
    print('
--- V2 RAPOR ---
')
    print(v2_report.read_text(encoding='utf-8'))


## 8. 2-5'li Feature Kombinasyon Tarama Sonuclari

In [ ]:
combo_top_path = DATA / 'feature_combo_search_top20.csv'
if combo_top_path.exists():
    combo = pd.read_csv(combo_top_path)
    display(combo.head(20))
else:
    combo = None
    print('Kombinasyon dosyasi bulunamadi:', combo_top_path)


In [ ]:
if combo is not None:
    plt.figure(figsize=(8,4))
    sns.barplot(data=combo.head(10), x='r2', y='features', palette='magma')
    plt.title('Top 10 Ozellik Kombinasyonu (2-5 feature)')
    plt.xlabel('R2')
    plt.ylabel('Feature kombinasyonu')
    plt.tight_layout()
    plt.show()



9. Genel Ozet

- Leakage temizligi sonrasi gercekci ana skor: ~0.60
- Coklu model benchmark'ta en iyi model: RandomForest (~0.606)
- Group-aware daha sikida skorlar dusuyor (daha gercekci)
- Kucuk feature kombinasyonlari tek basina full modele gore daha dusuk kaldi

Bu notebook, tum ara calismalari tek yerde gormek ve sunum almak icin hazirdir.